In [1]:
import torch
from model import Model
from train import train
from graph_generation import random_graph
import algorithms

#### Experimental setup

In [2]:
num_nodes = 6 # we will consider weighted undirected graph of 6 nodes in the notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Create the model

In [3]:
model = Model(['bfs', 'bf', 'prim', 'cc'], in_dim=1, hidden_dim=32, out_dim=1).to(device)

#### Train the model

In [4]:
train(model, train_size=100, val_size=20, num_nodes=6, epochs=15, patience=20)

Generating training data... Generated!
Start training process...
Epoch 1 | train loss = 0.80 (state=0.45, predec=0.23, term=1.18) | val loss = 0.44 (state=0.16, predec=0.14, term=1.06)
Epoch 2 | train loss = 0.40 (state=0.14, predec=0.14, term=0.98) | val loss = 0.39 (state=0.11, predec=0.13, term=1.08)
Epoch 3 | train loss = 0.36 (state=0.12, predec=0.12, term=0.90) | val loss = 0.31 (state=0.09, predec=0.12, term=0.85)
Epoch 4 | train loss = 0.34 (state=0.11, predec=0.11, term=0.87) | val loss = 0.32 (state=0.10, predec=0.11, term=0.86)
Epoch 5 | train loss = 0.34 (state=0.12, predec=0.10, term=0.85) | val loss = 0.34 (state=0.10, predec=0.10, term=0.97)
Epoch 6 | train loss = 0.32 (state=0.11, predec=0.09, term=0.82) | val loss = 0.31 (state=0.11, predec=0.09, term=0.78)
Epoch 7 | train loss = 0.34 (state=0.13, predec=0.09, term=0.83) | val loss = 0.31 (state=0.11, predec=0.10, term=0.75)
Epoch 8 | train loss = 0.33 (state=0.12, predec=0.08, term=0.81) | val loss = 0.31 (state=0.10,

#### Simulate Breadth-First-Search (BFS)

In [9]:
# generate graph
graph, source = random_graph(num_nodes=num_nodes, weighted=True, weight_mn=0.2, weight_mx=2.0, self_loop=True), 0

# generate bfs states
states, predecessors, inf, terminations = algorithms.compute_states('bfs', graph, source)
steps = algorithms.generate_steps(states, predecessors, terminations)

# input of the model prediction
edges = graph.get_edge_tensors(device)
h = torch.zeros(graph.num_nodes, model.hidden_dim, device=device)
x = torch.tensor(states[0], dtype=torch.float32).unsqueeze(1).to(device)

for step, (state, next_state, predecessors, termination) in enumerate(steps):
    state_pred, p_pred, h, term_pred = model('bfs', edges, x, h)

    # convert the state prediction into probabilities
    prediction = [1 if p > 0.5 else 0 for p in torch.sigmoid(state_pred).squeeze(1)]

    print(f"Step {step + 1}")
    print(f"Prediction: {prediction}")
    print(f"Target:     {next_state}")

    # termination prediction
    if torch.sigmoid(term_pred).item() > 0.5:
        break

    h = h.detach()
    x = torch.sigmoid(state_pred).detach() # output of previous iteration becomes input of the next

Step 1
Prediction: [1, 0, 0, 1, 0, 0]
Target:     [1, 0, 0, 1, 0, 0]
Step 2
Prediction: [1, 0, 1, 1, 0, 1]
Target:     [1, 0, 1, 1, 0, 1]
Step 3
Prediction: [1, 0, 1, 1, 1, 1]
Target:     [1, 0, 1, 1, 1, 1]


#### Simulate Bellman-Ford (BF)

In [12]:
# generate graph
graph, source = random_graph(num_nodes=num_nodes, weighted=True, weight_mn=0.2, weight_mx=2.0, self_loop=True), 0

# generate bf states
states, predecessors, inf, terminations = algorithms.compute_states('bf', graph, source)
steps = algorithms.generate_steps(states, predecessors, terminations)

# input of the model prediction
edges = graph.get_edge_tensors(device)
h = torch.zeros(graph.num_nodes, model.hidden_dim, device=device)
x = torch.tensor(states[0], dtype=torch.float32).unsqueeze(1).to(device)

for step, (state, next_state, predecessors, termination) in enumerate(steps):
    state_pred, p_pred, h, term_pred = model('bf', edges, x, h)

    prediction = [round(x.item(), 2) for x in state_pred.squeeze(1)]
    target = [round(x, 2) for x in next_state]

    print(f"Step {step + 1}")
    print(f"Prediction: {prediction}")
    print(f"Target:     {target}")

    # termination prediction
    if torch.sigmoid(term_pred).item() > 0.5:
        break

    h = h.detach()
    x = torch.sigmoid(state_pred).detach() # output of previous iteration becomes input of the next

Step 1
Prediction: [0.32, 0.85, 1.07, 0.5, 0.97, 2.21]
Target:     [0.0, 0.85, 1.27, 0.35, 1.08, 2.27]
